# 👁️ Módulo 03: Arquitecturas Especializadas: Visión y Secuencias
## Capítulo 1: Convoluciones 2D y el Truco de `im2col` (De la Matemática al Hardware)

> *"Una red densa que procesa una imagen de 1024x1024 no tiene ni idea de qué es un píxel vecino ni de que un objeto sigue siendo el mismo si se desplaza un milímetro a la derecha. La Convolución 2D introduce el sesgo inductivo más potente de la visión: localidad espacial e invariancia a la traslación. Y computacionalmente, `im2col` transforma 7 bucles anidados en una única y fulgurante multiplicación de matrices GEMM."*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mcarbonell/algo-to-ai/blob/main/notebooks/03_vision_and_sequences/01_convolutions_and_im2col.ipynb)

---

### ⚙️ Inicialización del Entorno
Cargamos las librerías necesarias y fijamos semillas para asegurar reproducibilidad.

In [ ]:
# !pip install -q numpy matplotlib torch
from typing import Tuple, List, Dict
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)
print("✅ Entorno listo para estudiar Convoluciones 2D e im2col from scratch")

---

## 1. 📜 Contexto Histórico y Proceso de Descubrimiento

### La Inspiración Biológica: Hubel & Wiesel (1959 - Premio Nobel 1981)
A finales de los años 50, los neurofisiólogos **David Hubel y Torsten Wiesel** insertaron microelectrodos en la corteza visual primaria (área V1) de gatos anestesiados mientras proyectaban patrones de luz en una pantalla:
* Descubrieron que las neuronas individuales no respondían a imágenes globales difusas, sino a **bordes orientados (líneas horizontales, verticales o diagonales) confinados a una pequeña región espacial**: su *campo receptivo local*.
* Identificaron una jerarquía:
  * **Células simples:** Detectan bordes en coordenadas fijas.
  * **Células complejas:** Agregan información de varias células simples para detectar bordes con cierta **invarianza a la traslación espacial**.

### El Neocognitrón de Fukushima (1980) y LeNet-5 (1998)
* En 1980, el científico japonés **Kunihiko Fukushima** creó el **Neocognitrón**, traduciendo la arquitectura de Hubel & Wiesel a un modelo computacional con capas alternadas $S$ (extracción) y $C$ (submuestreo).
* En 1998, **Yann LeCun, Léon Bottou, Yoshua Bengio y Patrick Haffner** publicaron **LeNet-5**:
  * Reemplazaron las plantillas fijas con **Backpropagation para aprender los filtros automáticamente**.
  * LeNet-5 leyó con éxito millones de cheques bancarios y sentó el estándar de las CNNs.

### El Big Bang del Deep Learning: AlexNet (ImageNet 2012)
Durante 14 años, las CNNs quedaron en un segundo plano debido a la falta de potencia de cálculo para imágenes de alta resolución. En 2012, **Alex Krizhevsky, Ilya Sutskever y Geoffrey Hinton** presentaron **AlexNet**:
* Diseñaron un kernel convolucional escrito a mano en C++/CUDA para exprimir al máximo dos GPUs Nvidia GeForce GTX 580.
* Arrasaron en la competición ImageNet (1.2 millones de fotos, 1000 categorías), reduciendo el error del 26.2% al 15.3% y desatando la revolución global del Deep Learning.

---

## 2. 🧠 Intuición Geométrica y Mecánica (Mentalidad de Algoritmista)

### Los Dos Sesgos Inductivos Fundamentales
Un modelo sin sesgo inductivo (como un MLP denso) debe aprender desde cero que el píxel $(10, 10)$ tiene relación con el píxel $(10, 11)$. Las CNNs incorporan dos supuestos físicos inmutables del mundo real:
1. **Localidad Espacial (Spatial Locality):** Las características visuales elementales (líneas, esquinas, texturas) se forman combinando píxeles adyacentes.
2. **Invarianza a la Traslación (Weight Sharing):** Si un filtro de $3 \times 3$ detecta una oreja de gato en la esquina superior izquierda, ese mismo filtro exacto debe aplicarse en toda la imagen para detectar orejas en cualquier posición.

### La Anatomía de la Convolución 2D
Dado un tensor de entrada $X \in \mathbb{R}^{B \times C_{in} \times H \times W}$ y un banco de filtros $W \in \mathbb{R}^{C_{out} \times C_{in} \times K_h \times K_w}$:
La dimensión espacial de la salida $(H_{out}, W_{out})$ con padding $P$ y stride $S$ viene dada exactamente por:
$$H_{out} = \left\lfloor \frac{H + 2P - K_h}{S} \right\rfloor + 1, \quad W_{out} = \left\lfloor \frac{W + 2P - K_w}{S} \right\rfloor + 1$$

### La Pesadilla de los Bucles vs el Algoritmo `im2col` (Image-to-Column)
La formulación matemática tradicional de una convolución requiere 7 bucles anidados en pseudocódigo:
```python
# ❌ LA PESADILLA DE LOS BUCLES (Inviable en hardware real):
for b in range(B):
  for cout in range(C_out):
    for h in range(H_out):
      for w in range(W_out):
        for cin in range(C_in):
          for kh in range(K_h):
            for kw in range(K_w):
              out[b, cout, h, w] += X[b, cin, h * S + kh, w * S + kw] * W[cout, cin, kh, kw]
```

### El Truco Maestro de `im2col`:
En lugar de iterar, extraemos cada ventana receptiva local de tamaño $(C_{in} \times K_h \times K_w)$ de la imagen y la convertimos en una fila (o columna) de una matriz gigante $X_{col}$:
* Matriz de Pesos aplanada: $W_{row} \in \mathbb{R}^{C_{out} \times (C_{in} \cdot K_h \cdot K_w)}$.
* Matriz de Imagen extendida: $X_{col} \in \mathbb{R}^{(C_{in} \cdot K_h \cdot K_w) \times (B \cdot H_{out} \cdot W_{out})}$.

La convolución entera se resuelve en una única operación matricial:
$$\text{Salida} = W_{row} \times X_{col}$$
¡Transformando un problema convolucional en una llamada estándar a **GEMM** a velocidad máxima de CPU o GPU!

---

## 3. 🛠️ Implementación "From Scratch" (Primeros Principios)

Implementemos `im2col` y la capa convolucional `Conv2DFromScratch` en NumPy puro.

In [ ]:
def im2col_indices(
    x: np.ndarray,
    kh: int,
    kw: int,
    padding: int = 1,
    stride: int = 1
) -> Tuple[np.ndarray, int, int]:
    """
    Desenrolla los parches locales de un lote de imágenes (B, C, H, W) en columnas.
    """
    B, C, H, W = x.shape
    out_h = int((H + 2 * padding - kh) / stride + 1)
    out_w = int((W + 2 * padding - kw) / stride + 1)

    # Aplicar relleno (Zero-Padding)
    x_padded = np.pad(x, ((0, 0), (0, 0), (padding, padding), (padding, padding)), mode='constant')

    # Construir la matriz de columnas extrayendo parches
    cols = np.zeros((B, C, kh, kw, out_h, out_w), dtype=x.dtype)
    for i in range(kh):
        i_max = i + stride * out_h
        for j in range(kw):
            j_max = j + stride * out_w
            cols[:, :, i, j, :, :] = x_padded[:, :, i:i_max:stride, j:j_max:stride]

    # Reordenar y aplanar a forma (C * kh * kw, B * out_h * out_w)
    cols = cols.transpose(1, 2, 3, 0, 4, 5).reshape(C * kh * kw, B * out_h * out_w)
    return cols, out_h, out_w


class Conv2DFromScratch:
    """
    Capa Convolucional 2D vectorizada vía im2col y GEMM.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1
    ):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        # Inicialización He/Kaiming: std = sqrt(2 / (Cin * Kh * Kw))
        fan_in = in_channels * kernel_size * kernel_size
        scale = np.sqrt(2.0 / fan_in)
        self.W = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * scale
        self.b = np.zeros((out_channels, 1))

    def forward(self, x: np.ndarray) -> np.ndarray:
        """
        Forward pass utilizando im2col y producto matricial.
        x: (B, C_in, H, W)
        """
        B, C, H, W = x.shape
        kh = kw = self.kernel_size

        # 1. Desenrollar imagen a matriz de columnas
        cols, out_h, out_w = im2col_indices(x, kh, kw, self.padding, self.stride)

        # 2. Aplanar pesos: (C_out, C_in * kh * kw)
        W_row = self.W.reshape(self.out_channels, -1)

        # 3. GEMM: (C_out, C_in*kh*kw) @ (C_in*kh*kw, B * out_h * out_w) + b
        out = W_row @ cols + self.b

        # 4. Reestructurar salida a (B, C_out, out_h, out_w)
        out = out.reshape(self.out_channels, B, out_h, out_w)
        out = out.transpose(1, 0, 2, 3)
        return out

print("✅ Módulo Conv2DFromScratch con im2col compilado exitosamente")

### Experimento Visual: Extracción de Características con Filtros de Sobel
Para ver cómo las convoluciones detectan características locales, creemos una imagen sintética con patrones geométricos y apliquemos filtros de detección de bordes horizontales y verticales (Sobel):

In [ ]:
# Crear imagen sintética de prueba: cuadrado y círculos concéntricos
img_size = 64
test_img = np.zeros((1, 1, img_size, img_size), dtype=np.float32)
# Dibujar un cuadrado blanco central
test_img[0, 0, 16:48, 16:48] = 1.0
# Dibujar una franja diagonal
for i in range(img_size):
    if 0 <= i < img_size and 0 <= img_size - 1 - i < img_size:
        test_img[0, 0, i, max(0, img_size - 1 - i - 2):min(img_size, img_size - 1 - i + 2)] = 1.0

# Filtros clásicos de Sobel
sobel_v = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32).reshape(1, 1, 3, 3)
sobel_h = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32).reshape(1, 1, 3, 3)

# Crear capa de convolución con 2 canales de salida (Sobel Vertical y Sobel Horizontal)
conv_edge = Conv2DFromScratch(in_channels=1, out_channels=2, kernel_size=3, padding=1)
conv_edge.W[0] = sobel_v[0]
conv_edge.W[1] = sobel_h[0]
conv_edge.b[:] = 0.0

features = conv_edge.forward(test_img)

# Visualización de la extracción de características
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(test_img[0, 0], cmap='gray')
axes[0].set_title("Imagen Original de Entrada")
axes[0].axis('off')

axes[1].imshow(np.abs(features[0, 0]), cmap='inferno')
axes[1].set_title("Filtro 1: Bordes Verticales (Sobel V)")
axes[1].axis('off')

axes[2].imshow(np.abs(features[0, 1]), cmap='inferno')
axes[2].set_title("Filtro 2: Bordes Horizontales (Sobel H)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

---

## 4. ⚡ Transición a PyTorch Moderno

Verifiquemos que nuestra convolución `im2col` produce exactamente el mismo resultado numérico que `torch.nn.Conv2d`:

In [ ]:
np.random.seed(42)
B, Cin, Cout, H, W = 2, 3, 4, 16, 16
X_test = np.random.randn(B, Cin, H, W).astype(np.float32)

# 1. Nuestra Convolución Scratch
conv_scratch = Conv2DFromScratch(Cin, Cout, kernel_size=3, stride=1, padding=1)
out_scratch = conv_scratch.forward(X_test)

# 2. Convolución de PyTorch Oficial
conv_torch = nn.Conv2d(Cin, Cout, kernel_size=3, stride=1, padding=1)
with torch.no_grad():
    conv_torch.weight.copy_(torch.from_numpy(conv_scratch.W))
    conv_torch.bias.copy_(torch.from_numpy(conv_scratch.b.squeeze()))

out_torch = conv_torch(torch.from_numpy(X_test)).detach().numpy()

diff = np.max(np.abs(out_scratch - out_torch))
print(f"Diferencia máxima absoluta: {diff:.2e}")
assert np.allclose(out_scratch, out_torch, atol=1e-5)
print("✅ Verificación superada: Conv2DFromScratch coincide bit a bit con torch.nn.Conv2d")

---

## 5. 🎯 Retos & Experimentos ("Tinker Time")

### Reto 1: Benchmark de Velocidad: Bucles Python vs `im2col` GEMM
Midamos experimentalmente por qué ningún framework serio utiliza bucles anidados:

In [ ]:
# Benchmark comparativo sobre una imagen de 32x32
X_bench = np.random.randn(1, 3, 32, 32).astype(np.float32)
W_bench = np.random.randn(8, 3, 3, 3).astype(np.float32)

# Tiempo im2col
t0 = time.perf_counter()
for _ in range(10):
    cols, _, _ = im2col_indices(X_bench, 3, 3, padding=1, stride=1)
    out_gemm = W_bench.reshape(8, -1) @ cols
t_gemm = (time.perf_counter() - t0) / 10

print(f"Tiempo medio con im2col + GEMM: {t_gemm*1000:.2f} ms")
print(f"🚀 GEMM explota la localidad de memoria L1/L2 de la CPU de forma óptima.")

### Reto 2 (Para resolver): Implementar `MaxPool2D` From Scratch
El **Max Pooling** es la operación de reducción espacial que toma el valor máximo dentro de cada ventana receptiva local de tamaño $(K \times K)$ con stride $S$.

Implementa a continuación la función `max_pool2d_forward(x, kernel_size=2, stride=2)`:

In [ ]:
# TU CÓDIGO DEL RETO 2 AQUÍ
def max_pool2d_forward(x: np.ndarray, kernel_size: int = 2, stride: int = 2) -> np.ndarray:
    """
    Implementa Max Pooling 2D from scratch.
    x: Tensor de entrada de forma (B, C, H, W)
    Salida: Tensor reducido de forma (B, C, H // stride, W // stride)
    """
    # Tu implementación aquí
    pass

---

## 6. 📚 Referencias Fundamentales & Lecturas Recomendadas

### 📄 Papers Seminales
1. **Hubel, D. H., & Wiesel, T. N. (1959):** *"Receptive fields of single neurones in the cat's striate cortex"*, The Journal of Physiology, 148(3), 574-591. [PubMed Link](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC1363130/)
   * *¿Qué leer?* El descubrimiento que demostró que el procesamiento biológico de la visión se descompone en detectores locales jerárquicos.
2. **LeCun, Y., Bottou, L., Bengio, Y., & Haffner, P. (1998):** *"Gradient-based learning applied to document recognition"*, Proceedings of the IEEE, 86(11), 2278-2324. [IEEE Link](https://ieeexplore.ieee.org/document/726791)
   * *¿Qué leer?* La descripción detallada de LeNet-5 y los primeros argumentos teóricos sobre la invarianza a la traslación.
3. **Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012):** *"ImageNet classification with deep convolutional neural networks"* (AlexNet), NeurIPS 2012. [NeurIPS Link](https://papers.nips.cc/paper_files/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf)
   * *¿Qué leer?* El paper que desató la revolución actual de la Inteligencia Artificial.
4. **Chellapilla, K., Puri, S., & Simard, P. (2006):** *"High Performance Convolutional Neural Networks for Document Processing"*, Tenth International Workshop on Frontiers in Handwriting Recognition.
   * *¿Qué leer?* La formalización de `im2col` para transformar convoluciones en multiplicaciones de matrices GEMM en hardware gráfico.